In [ ]:
#Eliyahu Venarchick
#10/31/2026
#OpenMc Be2C Moderator, NO clad, 19.75%wt percentage U235

# ===================================================================================================
# Initialization
# ===================================================================================================

import os
import openmc
#import matplotlib.pyplot as plt



library_path = "/home/n4sa_fan/openmc/Cross_Section_Libraries/endfb-viii.1-hdf5/cross_sections.xml"
os.environ['OPENMC_CROSS_SECTIONS'] = library_path

# ===================================================================================================
# Materials
# ===================================================================================================

fuel = openmc.Material(0,name = "UO2",temperature=300) #Fuel Enriched 19.75 wt% U235
fuel.add_nuclide("U234",4.098e-05,'ao')
fuel.add_nuclide("U235",4.605e-03,'ao')
fuel.add_nuclide("U236",2.118e-05,'ao')
fuel.add_nuclide("U238",1.856e-02,'ao')
fuel.set_density('g/cm3',10.38) 
fuel.add_element('O',2.0,'ao')


moderator = openmc.Material(name = 'Be2C',temperature=300)
moderator.set_density('g/cm3',1.90)
moderator.add_element('Be',2.0)
moderator.add_element('C',1.0)
moderator.add_s_alpha_beta("c_Be_in_Be2C")
moderator.add_s_alpha_beta("c_C_in_Be2C")

materials = openmc.Materials([fuel,moderator])

materials.cross_sections = library_path
materials.export_to_xml()

# ===================================================================================================
# Geometry
# ===================================================================================================
radius = 1.1
pitch = 2.5
fuel_OR = openmc.ZCylinder(r = radius, name = "Fuel Outer Radius")
box = openmc.model.RectangularPrism(pitch,pitch,boundary_type ='reflective')

pincell_universe = openmc.Universe(name = 'Fuel Pin Universe')

fuel_cell = openmc.Cell(name = "Fuel Cell",fill = fuel, region = -fuel_OR)
pincell_universe.add_cell(fuel_cell)

mod_cell = openmc.Cell(name = 'Moderator Cell',fill = moderator, region = +fuel_OR)
pincell_universe.add_cell(mod_cell)

root_cell = openmc.Cell(name = 'Root Cell',fill = pincell_universe)
root_cell.region = -box

root_universe = openmc.Universe(universe_id=1,name = "Root Universe")
root_universe.add_cell(root_cell)

geometry = openmc.Geometry(root_universe)
geometry.export_to_xml()

# ===================================================================================================
# Settings
# ===================================================================================================

settings = openmc.Settings()

settings.batches = 150
settings.inactive = 10
settings.particles = 5000

bounds = [-pitch/2, -pitch/2, -pitch/2, pitch/2, pitch/2, pitch/2]
uniform_dist = openmc.stats.Box(bounds[:3],bounds[3:],only_fissionable=True)
settings.source = openmc.IndependentSource(space = uniform_dist)
settings.export_to_xml()

# ===================================================================================================
#root_universe.plot(width =(pitch,pitch),basis='xy',colors={fuel_cell: 'green',mod_cell:'blue'})
#plt.title(f'radius = {radius}, pitch = {pitch}')

openmc.run()
